# EMI and Affordability Walkthrough — Demonstration

**This notebook uses a SYNTHETIC per-city dataset** (`data/synthetic_city_housing_baseline.csv`), generated by `examples/generate_synthetic_city_housing.py`. It is not real market or survey data. Every number below describes the synthetic generator's output, not a real Indian city — the point is to walk through and verify the EMI/affordability formulas and the toolkit's API, not to report an empirical finding.

In [1]:
import sys
from pathlib import Path

# Make the local src/ package importable without an editable install.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from housing_affordability.metrics import emi, affordability_summary, total_interest_paid, interest_burden_ratio
from housing_affordability.scenarios import run_standard_scenarios, STANDARD_SCENARIOS
from housing_affordability.data_loader import load_synthetic_city_baseline

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Verifying the EMI formula by hand

The standard amortizing-loan EMI formula is:

```
EMI = P * r * (1 + r)^n / ((1 + r)^n - 1)
```

where `r` is the *monthly* interest rate and `n` the number of monthly instalments. Let's compute one example by hand, step by step, and check it matches `emi()`.

In [2]:
# Hand-computed example: a 1,000,000 loan, 9% annual rate, 20-year tenure.
P = 1_000_000.0
annual_rate = 0.09
years = 20

r = annual_rate / 12          # monthly rate
n = years * 12                # number of monthly instalments

growth = (1 + r) ** n
emi_by_hand = P * r * growth / (growth - 1)

emi_from_function = emi(P, annual_rate, years)

print(f"monthly rate r         = {r:.6f}")
print(f"number of instalments n = {n}")
print(f"EMI (hand-computed)    = {emi_by_hand:,.4f}")
print(f"EMI (from function)    = {emi_from_function:,.4f}")
assert abs(emi_by_hand - emi_from_function) < 1e-9, "Mismatch between hand-computed and function EMI!"
print("\nMatch confirmed.")

monthly rate r         = 0.007500
number of instalments n = 240
EMI (hand-computed)    = 8,997.2596
EMI (from function)    = 8,997.2596

Match confirmed.


## 2. The zero-interest edge case

At `r = 0` the annuity formula is `0/0`. The function handles this explicitly: with no interest, the loan is just the principal split evenly across `n` instalments, `EMI = P / n`.

In [3]:
zero_rate_emi = emi(1_200_000, 0.0, 10)
expected = 1_200_000 / (10 * 12)
print(f"EMI at 0% interest, 10y on a 1,200,000 loan: {zero_rate_emi:,.2f}")
print(f"Expected (P / n):                            {expected:,.2f}")
assert abs(zero_rate_emi - expected) < 1e-9

EMI at 0% interest, 10y on a 1,200,000 loan: 10,000.00
Expected (P / n):                            10,000.00


## 3. Total interest and interest burden

Total interest paid over the life of the loan is `EMI * n - P`. The interest burden ratio is the share of total repayments (`principal + interest`) that is interest. Holding the rate fixed, a longer tenure means a *larger* interest burden, even though the monthly EMI itself is smaller.

In [4]:
for tenure in [10, 15, 20, 25, 30]:
    monthly = emi(P, annual_rate, tenure)
    interest = total_interest_paid(P, annual_rate, tenure)
    total_paid = interest + P
    burden = interest_burden_ratio(interest, total_paid)
    print(f"{tenure:>2}y tenure -> EMI={monthly:>10,.0f}  total interest={interest:>14,.0f}  interest burden={burden:.1%}")

10y tenure -> EMI=    12,668  total interest=       520,109  interest burden=34.2%
15y tenure -> EMI=    10,143  total interest=       825,680  interest burden=45.2%
20y tenure -> EMI=     8,997  total interest=     1,159,342  interest burden=53.7%
25y tenure -> EMI=     8,392  total interest=     1,517,589  interest burden=60.3%
30y tenure -> EMI=     8,046  total interest=     1,896,641  interest burden=65.5%


## 4. The synthetic per-city baseline

**Reminder: every number below is SYNTHETIC**, generated by `examples/generate_synthetic_city_housing.py` (seed=2024). The per-city anchors were hand-picked only to echo widely-reported *qualitative* relative orderings (Mumbai and Delhi NCR least affordable, Kolkata and Ahmedabad relatively more affordable among these eight) — they are not sourced statistics. See `docs/DATA_SOURCES.md` for where real data could be sourced.

In [5]:
baseline = load_synthetic_city_baseline()
baseline

,city,synthetic_annual_household_income_inr,synthetic_house_price_inr,illustrative_price_to_income_anchor,data_status,synthetic_price_to_income_ratio
0,Ahmedabad,"1,113,000.00","5,932,000.00",5.00,SYNTHETIC,5.33
1,Bengaluru,"1,614,000.00","12,407,000.00",8.00,SYNTHETIC,7.69
2,Chennai,"1,150,000.00","6,918,000.00",6.00,SYNTHETIC,6.02
3,Delhi NCR,"1,539,000.00","14,131,000.00",9.00,SYNTHETIC,9.18
4,Hyderabad,"1,392,000.00","9,318,000.00",6.50,SYNTHETIC,6.69
5,Kolkata,"1,040,000.00","5,046,000.00",5.00,SYNTHETIC,4.85
6,Mumbai,"1,566,000.00","18,251,000.00",11.00,SYNTHETIC,11.65
7,Pune,"1,262,000.00","9,120,000.00",7.00,SYNTHETIC,7.23


## 5. Affordability summary for one synthetic city

`affordability_summary()` bundles every metric for one city/scenario into a single object.

In [6]:
mumbai = baseline.set_index("city").loc["Mumbai"]

summary = affordability_summary(
    house_price=float(mumbai["synthetic_house_price_inr"]),
    annual_household_income=float(mumbai["synthetic_annual_household_income_inr"]),
    down_payment_pct=0.20,
    annual_interest_rate=0.085,
    loan_years=20,
    city="Mumbai",
)

for field in [
    "price_to_income", "monthly_emi", "mortgage_payment_to_income",
    "loan_to_value", "total_interest", "interest_burden", "years_to_save_down_payment",
]:
    print(f"{field:30s} {getattr(summary, field):,.3f}")

price_to_income                11.655
monthly_emi                    126,709.135
mortgage_payment_to_income     0.971
loan_to_value                  0.800
total_interest                 15,809,392.318
interest_burden                0.520
years_to_save_down_payment     11.655


## 6. Affordability summary across all eight synthetic cities

Same financing assumptions (20% down, 8.5% rate, 20-year tenure) applied to every city's synthetic baseline, so only the starting price/income differ.

In [7]:
rows = []
for _, row in baseline.iterrows():
    s = affordability_summary(
        house_price=float(row["synthetic_house_price_inr"]),
        annual_household_income=float(row["synthetic_annual_household_income_inr"]),
        down_payment_pct=0.20,
        annual_interest_rate=0.085,
        loan_years=20,
        city=row["city"],
    )
    rows.append({
        "city": s.city,
        "price_to_income": s.price_to_income,
        "monthly_emi": s.monthly_emi,
        "mortgage_payment_to_income": s.mortgage_payment_to_income,
        "years_to_save_down_payment": s.years_to_save_down_payment,
    })

comparison = pd.DataFrame(rows).sort_values("price_to_income", ascending=False).reset_index(drop=True)
comparison

,city,price_to_income,monthly_emi,mortgage_payment_to_income,years_to_save_down_payment
0,Mumbai,11.65,"126,709.13",0.97,11.65
1,Delhi NCR,9.18,"98,105.68",0.76,9.18
2,Bengaluru,7.69,"86,136.66",0.64,7.69
3,Pune,7.23,"63,316.38",0.60,7.23
4,Hyderabad,6.69,"64,691.02",0.56,6.69
5,Chennai,6.02,"48,028.81",0.50,6.02
6,Ahmedabad,5.33,"41,183.42",0.44,5.33
7,Kolkata,4.85,"35,032.29",0.40,4.85


Note the ordering above (Mumbai and Delhi NCR at the top, Kolkata and Ahmedabad at the bottom) was **built into the synthetic generator's anchors by construction** — see `examples/generate_synthetic_city_housing.py`. This cell demonstrates that the toolkit's ratio calculations correctly preserve that input ordering; it is not a finding the toolkit discovered.

## 7. Multi-year scenario projection

Three standard scenarios (conservative / baseline / optimistic) compound house-price growth, income growth, and a constant mortgage rate forward and report the first year in which accumulated savings meet the (growing) down-payment target. This is a **mechanical projection of the assumptions**, not a forecast.

In [8]:
for key, scenario in STANDARD_SCENARIOS.items():
    print(f"{scenario.name:12s} house_price_growth={scenario.house_price_growth:.1%}  "
          f"income_growth={scenario.income_growth:.1%}  mortgage_rate={scenario.mortgage_rate:.1%}  "
          f"savings_rate={scenario.savings_rate:.0%}")

Conservative house_price_growth=9.0%  income_growth=6.0%  mortgage_rate=9.5%  savings_rate=15%
Baseline     house_price_growth=7.0%  income_growth=8.0%  mortgage_rate=8.5%  savings_rate=20%
Optimistic   house_price_growth=5.0%  income_growth=10.0%  mortgage_rate=7.5%  savings_rate=25%


In [9]:
bengaluru = baseline.set_index("city").loc["Bengaluru"]

results = run_standard_scenarios(
    initial_house_price=float(bengaluru["synthetic_house_price_inr"]),
    initial_annual_income=float(bengaluru["synthetic_annual_household_income_inr"]),
    horizon_years=15,
)

for key, projection in results.items():
    years = projection.years_to_afford_down_payment
    label = f"{years} years" if years is not None else "> 15 years"
    final_pir = projection.table.iloc[-1]["price_to_income"]
    print(f"{projection.scenario.name:12s} years to afford 20% down payment: {label:>10s}   "
          f"PIR after 15y: {final_pir:.2f}x")

Conservative years to afford 20% down payment: > 15 years   PIR after 15y: 11.68x
Baseline     years to afford 20% down payment:   10 years   PIR after 15y: 6.69x
Optimistic   years to afford 20% down payment:    6 years   PIR after 15y: 3.83x


In [10]:
results["baseline"].table.head(8)

,house_price,annual_income,monthly_income,down_payment_target,cumulative_savings,down_payment_met,loan_amount,monthly_emi,price_to_income,mortgage_payment_to_income
year,,,,,,,,,,
1,"13,275,490.00","1,743,120.00","145,260.00","2,655,098.00","348,624.00",False,"10,620,392.00","92,166.23",7.62,0.63
2,"14,204,774.30","1,882,569.60","156,880.80","2,840,954.86","725,137.92",False,"11,363,819.44","98,617.87",7.55,0.63
3,"15,199,108.50","2,033,175.17","169,431.26","3,039,821.70","1,131,772.95",False,"12,159,286.80","105,521.12",7.48,0.62
4,"16,263,046.10","2,195,829.18","182,985.77","3,252,609.22","1,570,938.79",False,"13,010,436.88","112,907.59",7.41,0.62
5,"17,401,459.32","2,371,495.52","197,624.63","3,480,291.86","2,045,237.89",False,"13,921,167.46","120,811.13",7.34,0.61
6,"18,619,561.48","2,561,215.16","213,434.60","3,723,912.30","2,557,480.92",False,"14,895,649.18","129,267.90",7.27,0.61
7,"19,922,930.78","2,766,112.37","230,509.36","3,984,586.16","3,110,703.40",False,"15,938,344.62","138,316.66",7.20,0.60
8,"21,317,535.93","2,987,401.36","248,950.11","4,263,507.19","3,708,183.67",False,"17,054,028.75","147,998.82",7.14,0.59


## Summary

- The EMI formula implemented in `housing_affordability.metrics.emi` was verified against a hand-computed closed-form example and matches to floating-point precision.
- The zero-interest edge case is handled explicitly rather than dividing by zero.
- `affordability_summary` correctly bundles every ratio for a city/scenario, and preserves the (by-construction) relative ordering of the synthetic city baseline.
- `run_standard_scenarios` projects three different growth-assumption sets forward and reports when a down-payment target becomes affordable — a mechanical consequence of the chosen growth rates, demonstrated here on synthetic inputs only.

See `docs/METHODOLOGY.md` for full derivations and `docs/DATA_SOURCES.md` for where real data could be sourced for a future version.